# 04 — Canonical analysis tables (Clean)

This notebook computes the **canonical question-level analysis outputs** from the processed wide table (N=25 questions × 3 conditions).  
It intentionally produces **machine-readable canonical tables** that are later formatted into manuscript-ready figures/tables in **Notebook 05**.

**Unit of analysis:** question (N=25).  
**Interpretation:** descriptive comparisons across task instances (questions), not population inference over participants.

## Inputs
- `../data/processed/buddi_paper_question_eval_wide.csv`

## Outputs
Written to `../outputs/buddi_paper/v1/analysis/`:
- `tables/` — canonical analysis tables (CSV)
- `logs/` — QA summaries + run manifest

## Step 0 — Environment, configuration, and output paths

We set:
- the condition order (**raw → struct → sem**) for consistent reporting,
- the bootstrap settings (10,000 resamples; fixed seed),
- stable output directories for canonical tables and logs.

In [21]:
# Imports, paths and constraints and load
# --------------------------

import json, hashlib
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime

from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.contingency_tables import mcnemar
from statsmodels.stats.multitest import multipletests


PAPER_SLUG = "buddi_paper"
RELEASE_ID = "v1"
RUN_ID = "v1_clean_04_canonical"

SEED = 20260106
N_BOOT = 10_000

CONDS = ["raw", "struct", "sem"]
COND_ORDER = ["raw", "struct", "sem"]
PAIRS = [("raw","struct"), ("struct","sem"), ("raw","sem")]

WIDE_PATH = f"../data/processed/{PAPER_SLUG}_question_eval_wide.csv"

BASE_OUT = Path(f"../outputs/{PAPER_SLUG}/{RELEASE_ID}/analysis")
OUT_TAB = BASE_OUT / "tables"
OUT_LOG = BASE_OUT / "logs"
OUT_TAB.mkdir(parents=True, exist_ok=True)
OUT_LOG.mkdir(parents=True, exist_ok=True)

NOW = datetime.now().isoformat(timespec="seconds")


## Step 1 — Load the wide table and validate integrity

We validate assumptions required for paired, question-level analyses:

- Exactly **25 rows** and **25 unique question_id**
- Exactly **5 questions per complexity level** (levels 1–5)
- `correct_*` columns are **binary (0/1)**
- `tokens_*` and `TTFT_*` are numeric and complete (TTFT assumed complete for this pipeline)

If any validation fails, we **raise an error** to prevent silent pairing mistakes.

In [22]:
df = pd.read_csv(WIDE_PATH)

# hard validations
if len(df) != 25:
    raise ValueError(f"Expected 25 rows; found {len(df)}")
if df["question_id"].nunique() != 25:
    raise ValueError("Expected 25 unique question_id.")
cnt = df["complexity_level"].value_counts().sort_index()
for lvl in [1,2,3,4,5]:
    if int(cnt.get(lvl,0)) != 5:
        raise ValueError(f"Expected 5 questions at complexity {lvl}; got {cnt.to_dict()}")

for c in CONDS:
    vals = set(df[f"correct_{c}"].dropna().unique())
    if not vals.issubset({0,1}):
        raise ValueError(f"correct_{c} not binary 0/1: {vals}")

for c in CONDS:
    df[f"tokens_{c}"] = pd.to_numeric(df[f"tokens_{c}"], errors="coerce")
    df[f"TTFT_{c}"] = pd.to_numeric(df[f"TTFT_{c}"], errors="coerce")
    if df[f"tokens_{c}"].isna().any():
        raise ValueError(f"Non-numeric tokens_{c}.")
    if df[f"TTFT_{c}"].isna().any():
        raise ValueError(f"Non-numeric TTFT_{c} (TTFT assumed complete).")

ALLOWED_ERROR_CATS = {"none","structural","semantic","computational"}

for c in CONDS:
    col = f"error_category_{c}"
    if col not in df.columns:
        raise ValueError(f"Missing column: {col}")
    vals = set(df[col].astype(str).str.strip().str.lower().unique())
    bad = vals - ALLOWED_ERROR_CATS
    if bad:
        raise ValueError(f"Unexpected {col} values: {bad}. Allowed: {ALLOWED_ERROR_CATS}")


print("Loaded and validated:", WIDE_PATH)

Loaded and validated: ../data/processed/buddi_paper_question_eval_wide.csv


## Step 2 — Define analysis helper functions

We define reusable functions for:

- **Wilson** 95% confidence intervals (proportions)
- **McNemar** paired tests (with discordant counts)
- **Bootstrap** confidence intervals for paired differences/ratios (resampling questions)
- Median + mean + IQR summaries
- Process-variable normalization (yes/no/NA)
- Robust 2×2 linkage testing (handles degenerate tables where a p-value is not defined)

These helpers keep the analysis consistent across outcomes.

In [23]:
# Helpers
# --------------------------
cond_cat = pd.CategoricalDtype(categories=COND_ORDER, ordered=True)
pair_cat = pd.CategoricalDtype(categories=["raw_vs_struct","struct_vs_sem","raw_vs_sem"], ordered=True)
pair_cat2 = pd.CategoricalDtype(categories=["raw_minus_struct","struct_minus_sem","raw_minus_sem"], ordered=True)

def wilson_ci(x:int, n:int, alpha=0.05):
    if n == 0:
        return (np.nan, np.nan)
    lo, hi = proportion_confint(count=x, nobs=n, alpha=alpha, method="wilson")
    return float(lo), float(hi)

def mcnemar_with_improve_worsen(a: pd.Series, b: pd.Series):
    """
    a, b are 0/1 correctness. We label discordants as:
      improved = (a=0, b=1)
      worsened = (a=1, b=0)
    """
    tmp = pd.DataFrame({"a": a, "b": b}).dropna()
    a = tmp["a"].astype(int).values
    b = tmp["b"].astype(int).values

    improved = int(((a==0)&(b==1)).sum())  # n01
    worsened = int(((a==1)&(b==0)).sum())  # n10
    n11 = int(((a==1)&(b==1)).sum())
    n00 = int(((a==0)&(b==0)).sum())

    table = np.array([[n00, improved],
                      [worsened, n11]])
    discordant = improved + worsened

    exact = discordant < 25
    res = mcnemar(table, exact=exact, correction=not exact)

    return float(res.pvalue), {
        "n00": n00, "n11": n11,
        "improved": improved, "worsened": worsened,
        "discordant": discordant,
        "used_exact": bool(exact)
    }

def bootstrap_ci(vals, alpha=0.05):
    lo = float(np.percentile(vals, 100*alpha/2))
    hi = float(np.percentile(vals, 100*(1-alpha/2)))
    return lo, hi

def bootstrap_paired(a: pd.Series, b: pd.Series, stat: str, n_boot=N_BOOT, seed=SEED):
    tmp = pd.DataFrame({"a": a, "b": b}).dropna()
    if stat == "median_ratio":
        tmp = tmp[(tmp["b"] > 0) & (tmp["a"] >= 0)]
    n = len(tmp)
    if n == 0:
        return (np.nan, np.nan, np.nan, 0)
    av = tmp["a"].astype(float).values
    bv = tmp["b"].astype(float).values
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, n, size=(n_boot, n))
    if stat == "mean_diff":
        boots = av[idx].mean(axis=1) - bv[idx].mean(axis=1)
        point = float(av.mean() - bv.mean())
    elif stat == "median_diff":
        boots = np.median(av[idx] - bv[idx], axis=1)
        point = float(np.median(av - bv))
    elif stat == "median_ratio":
        boots = np.median(av[idx] / bv[idx], axis=1)
        point = float(np.median(av / bv))
    else:
        raise ValueError("stat must be mean_diff|median_diff|median_ratio")
    lo, hi = bootstrap_ci(boots)
    return point, lo, hi, n

def median_iqr(s: pd.Series):
    s = s.dropna().astype(float)
    if len(s) == 0:
        return (np.nan, np.nan, np.nan, 0)
    med = float(np.median(s))
    q1 = float(np.percentile(s, 25))
    q3 = float(np.percentile(s, 75))
    return med, q1, q3, int(len(s))

def mean_sd(s: pd.Series):
    x = pd.to_numeric(s, errors="coerce").dropna().astype(float)
    n = int(len(x))
    if n == 0:
        return (np.nan, np.nan, 0)
    mu = float(x.mean())
    sd = float(x.std(ddof=1)) if n >= 2 else np.nan  # sample SD
    return (mu, sd, n)


YES = {"yes","y","true","t","1"}
NO  = {"no","n","false","f","0"}
NA  = {"na","n/a","nan","", "none"}

def norm_yes_no_na(s: pd.Series):
    s2 = s.astype(str).str.strip().str.lower()
    s2 = s2.replace({"n/a":"na","nan":"na", "": "na"})
    def _map(v):
        if v in YES: return "yes"
        if v in NO: return "no"
        if v in NA: return "na"
        return v
    return s2.map(_map)

def rate_ci(s_norm: pd.Series):
    classif = s_norm.isin(["yes","no"])
    n = int(classif.sum())
    if n == 0:
        return {"n_classifiable":0,"n_yes":0,"rate":np.nan,"wilson95_low":np.nan,"wilson95_high":np.nan}
    n_yes = int((s_norm[classif] == "yes").sum())
    lo, hi = wilson_ci(n_yes, n)
    return {"n_classifiable":n,"n_yes":n_yes,"rate":n_yes/n,"wilson95_low":lo,"wilson95_high":hi}

def link_2x2_fisher(correct: pd.Series, proc_norm: pd.Series):
    """
    Within-condition exploratory link:
      correct (0/1) vs proc_yes (yes/no)
    Use Fisher exact (two-sided) always.
    """
    m = proc_norm.isin(["yes","no"]) & correct.notna()
    d = pd.DataFrame({"correct": correct[m].astype(int),
                      "proc_yes": (proc_norm[m]=="yes").astype(int)})
    if len(d) == 0:
        return {"n":0,"method":"skipped_no_classifiable","p_value":np.nan}

    tab = pd.crosstab(d["proc_yes"], d["correct"])
    for r in [0,1]:
        if r not in tab.index: tab.loc[r] = 0
    for c in [0,1]:
        if c not in tab.columns: tab[c] = 0
    tab = tab.sort_index().sort_index(axis=1)
    table = tab.values.astype(int)

    out = {
        "n": int(len(d)),
        "table_proc_no_incorrect": int(table[0,0]),
        "table_proc_no_correct": int(table[0,1]),
        "table_proc_yes_incorrect": int(table[1,0]),
        "table_proc_yes_correct": int(table[1,1]),
        "method": "fisher_exact_2x2",
    }

    n_yes = int(table[1,:].sum())
    n_no  = int(table[0,:].sum())
    out["p_correct_given_proc_yes"] = (table[1,1]/n_yes) if n_yes else np.nan
    out["p_correct_given_proc_no"]  = (table[0,1]/n_no) if n_no else np.nan

    row_sums = table.sum(axis=1)
    col_sums = table.sum(axis=0)
    if (row_sums == 0).any() or (col_sums == 0).any():
        out["p_value"] = np.nan
        out["method"] = "skipped_degenerate_table"
        return out

    _, p = stats.fisher_exact(table, alternative="two-sided")
    out["p_value"] = float(p)
    return out

def paired_mcnemar_process(a_norm: pd.Series, b_norm: pd.Series):
    """
    Paired McNemar for process variables (yes/no only).
    Returns discordants labeled directionally:
      improved = (A=no, B=yes)
      worsened = (A=yes, B=no)
    """
    m = a_norm.isin(["yes","no"]) & b_norm.isin(["yes","no"])
    n_pairs = int(m.sum())
    if n_pairs == 0:
        return {"n_pairs":0,"mcnemar_p":np.nan,"improved":np.nan,"worsened":np.nan,"n00":np.nan,"n11":np.nan,"used_exact":np.nan}

    a_bin = (a_norm[m] == "yes").astype(int)
    b_bin = (b_norm[m] == "yes").astype(int)

    # improved/worsened relative to A -> B
    improved = int(((a_bin==0) & (b_bin==1)).sum())  # n01
    worsened = int(((a_bin==1) & (b_bin==0)).sum())  # n10
    n11 = int(((a_bin==1) & (b_bin==1)).sum())
    n00 = int(((a_bin==0) & (b_bin==0)).sum())

    table = np.array([[n00, improved],
                      [worsened, n11]])
    discordant = improved + worsened
    exact = discordant < 25
    res = mcnemar(table, exact=exact, correction=not exact)

    return {
        "n_pairs": n_pairs,
        "mcnemar_p": float(res.pvalue),
        "improved": improved,
        "worsened": worsened,
        "n00": n00,
        "n11": n11,
        "used_exact": bool(exact),
    }


def normalize_cat(s: pd.Series):
    return (s.astype(str).str.strip().str.lower()
            .replace({"nan":np.nan,"n/a":np.nan,"na":np.nan})
            .fillna("missing"))

def sha256_file(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8192), b""):
            h.update(chunk)
    return h.hexdigest()

def clopper_pearson_ci(x: int, n: int, alpha=0.05):
    if n == 0:
        return (np.nan, np.nan)
    lo, hi = proportion_confint(count=x, nobs=n, alpha=alpha, method="beta")
    return float(lo), float(hi)


## Step 3 — Accuracy (primary): McNemar + Holm correction + paired risk difference

We report overall accuracy per condition (n/N, %, Wilson 95% CI) and accuracy stratified by complexity (Wilson 95% CI; n=5 per level).

**Primary paired comparisons (3 prespecified):**
- raw vs struct
- struct vs sem
- raw vs sem

For each comparison:
- Discordant counts are labeled as:
  - **improved** = (A incorrect, B correct)
  - **worsened** = (A correct, B incorrect)
- McNemar p-value is computed (exact when sparse; otherwise continuity-corrected).
- We apply **Holm correction** across these 3 McNemar p-values and report:
  - unadjusted p
  - Holm-adjusted p

**Effect size: paired risk difference**
We estimate the paired risk difference using bootstrap resampling over questions (10,000 resamples; seed=20260106):
- **mean(correct_B − correct_A)** with 95% percentile CI

In [24]:
# Accuracy
# -----------------------
# Accuracy: overall + by complexity
# -----------------------
acc_rows = []
for c in CONDS:
    x = int(df[f"correct_{c}"].sum())
    n = int(df[f"correct_{c}"].notna().sum())
    lo, hi = wilson_ci(x, n)
    acc_rows.append({
        "condition": c, "n_correct": x, "n_questions": n,
        "accuracy": x/n, "accuracy_pct": 100*(x/n),
        "wilson95_low": lo, "wilson95_high": hi
    })
accuracy_overall = pd.DataFrame(acc_rows)
accuracy_overall["condition"] = accuracy_overall["condition"].astype(cond_cat)
accuracy_overall = accuracy_overall.sort_values("condition").reset_index(drop=True)

rows = []
for lvl, g in df.groupby("complexity_level", sort=True):
    for c in CONDS:
        x = int(g[f"correct_{c}"].sum())
        n = int(g[f"correct_{c}"].notna().sum())
        lo, hi = wilson_ci(x, n)
        rows.append({
            "complexity_level": int(lvl), "condition": c,
            "n_correct": x, "n_questions": n,
            "accuracy": x/n, "accuracy_pct": 100*(x/n),
            "wilson95_low": lo, "wilson95_high": hi
        })
accuracy_by_complexity = pd.DataFrame(rows)
accuracy_by_complexity["condition"] = accuracy_by_complexity["condition"].astype(cond_cat)
accuracy_by_complexity = accuracy_by_complexity.sort_values(["complexity_level","condition"]).reset_index(drop=True)

# -----------------------
# Accuracy paired tests: McNemar + Holm adjustment (primary family)
# -----------------------
mcn_rows = []
pvals = []
for a,b in PAIRS:
    p, counts = mcnemar_with_improve_worsen(df[f"correct_{a}"], df[f"correct_{b}"])
    mcn_rows.append({"comparison": f"{a}_vs_{b}", "p_unadj": p, **counts})
    pvals.append(p)

# Holm adjustment across the 3 prespecified comparisons
reject, p_holm, _, _ = multipletests(pvals, alpha=0.05, method="holm")

mcnemar_results = pd.DataFrame(mcn_rows)
mcnemar_results["p_holm"] = p_holm
mcnemar_results["comparison"] = mcnemar_results["comparison"].astype(pair_cat)
mcnemar_results = mcnemar_results.sort_values("comparison").reset_index(drop=True)

# -----------------------
# Accuracy effect size: paired risk difference (mean(B − A)) + bootstrap CI
# -----------------------
rd_rows = []
for a,b in PAIRS:
    # paired risk difference = mean(correct_b - correct_a)
    rd, lo, hi, n_pairs = bootstrap_paired(df[f"correct_{b}"], df[f"correct_{a}"], stat="mean_diff", seed=SEED)
    rd_rows.append({
        "comparison": f"{b}_minus_{a}",
        "n_pairs": n_pairs,
        "paired_risk_difference": rd,
        "boot95_low": lo,
        "boot95_high": hi
    })

bootstrap_risk_diffs = pd.DataFrame(rd_rows)
bootstrap_risk_diffs["comparison"] = bootstrap_risk_diffs["comparison"].astype(
    pd.CategoricalDtype(categories=["struct_minus_raw","sem_minus_struct","sem_minus_raw"], ordered=True)
)
bootstrap_risk_diffs = bootstrap_risk_diffs.sort_values("comparison").reset_index(drop=True)

display(accuracy_overall)
display(accuracy_by_complexity.head())
display(mcnemar_results)
display(bootstrap_risk_diffs)



,condition,n_correct,n_questions,accuracy,accuracy_pct,wilson95_low,wilson95_high
0,raw,8,25,0.32,32.0,0.172052,0.515897
1,struct,10,25,0.40,40.0,0.234033,0.592605
2,sem,24,25,0.96,96.0,0.804559,0.992904


,complexity_level,condition,n_correct,n_questions,accuracy,accuracy_pct,wilson95_low,wilson95_high
0,1,raw,5,5,1.0,100.0,0.565518,1.000000
1,1,struct,5,5,1.0,100.0,0.565518,1.000000
2,1,sem,5,5,1.0,100.0,0.565518,1.000000
3,2,raw,1,5,0.2,20.0,0.036224,0.624465
4,2,struct,1,5,0.2,20.0,0.036224,0.624465


,comparison,p_unadj,n00,n11,improved,worsened,discordant,used_exact,p_holm
0,raw_vs_struct,0.500000,15,8,2,0,2,True,0.500000
1,struct_vs_sem,0.000122,1,10,14,0,14,True,0.000244
2,raw_vs_sem,0.000031,1,8,16,0,16,True,0.000092


,comparison,n_pairs,paired_risk_difference,boot95_low,boot95_high
0,struct_minus_raw,25,0.08,0.00,0.20
1,sem_minus_struct,25,0.56,0.36,0.76
2,sem_minus_raw,25,0.64,0.44,0.84


## Step 4 — Efficiency (secondary): tokens and TTFT (bootstrap contrasts)

For each metric (tokens, TTFT) we report:
- Median (IQR) per condition (overall and by complexity).

Paired bootstrap contrasts are computed over questions (10,000 resamples; seed=20260106) for each comparison:
- **median paired difference** = median(metric_B − metric_A)
- **median of per-question ratios** = median(metric_B / metric_A)  
  *(computed per question; not ratio of medians)*

We report CIs for TTFT but do **not** perform hypothesis tests for TTFT.

In [25]:
def efficiency_tables(prefix: str, metric_label: str):
    overall, by, paired = [], [], []

    for c in CONDS:
        med, q1, q3, n = median_iqr(df[f"{prefix}_{c}"])
        mu, sd, n2 = mean_sd(df[f"{prefix}_{c}"])
        overall.append({
            "metric": metric_label, "stratum": "overall", "condition": c, "n": n,
            "median": med, "q1": q1, "q3": q3, "mean": mu, "sd": sd,
            "total_sum": float(pd.to_numeric(df[f"{prefix}_{c}"], errors="coerce").sum())
        })


    for lvl, g in df.groupby("complexity_level", sort=True):
        for c in CONDS:
            med, q1, q3, n = median_iqr(g[f"{prefix}_{c}"])
            by.append({"metric": metric_label, "stratum": f"complexity_{int(lvl)}",
                       "complexity_level": int(lvl), "condition": c, "n": n,
                       "median": med, "q1": q1, "q3": q3})

    strata = [("overall", df)] + [(f"complexity_{int(l)}", gg) for l,gg in df.groupby("complexity_level", sort=True)]
    for stratum_name, g in strata:
        for a,b in PAIRS:
            # We define contrasts as B - A and B / A (per question)
            md, md_lo, md_hi, n1 = bootstrap_paired(g[f"{prefix}_{b}"], g[f"{prefix}_{a}"], "median_diff", seed=SEED)
            mr, mr_lo, mr_hi, n2 = bootstrap_paired(g[f"{prefix}_{b}"], g[f"{prefix}_{a}"], "median_ratio", seed=SEED)
            paired.append({
                "metric": metric_label,
                "stratum": stratum_name,
                "comparison": f"{b}_minus_{a}",
                "median_paired_diff_B_minus_A": md,
                "diff_boot95_low": md_lo,
                "diff_boot95_high": md_hi,
                "n_pairs_diff": n1,
                "median_of_per_question_ratios_B_over_A": mr,
                "ratio_boot95_low": mr_lo,
                "ratio_boot95_high": mr_hi,
                "n_pairs_ratio": n2
            })

    overall = pd.DataFrame(overall)
    by = pd.DataFrame(by)
    paired = pd.DataFrame(paired)

    overall["condition"] = overall["condition"].astype(cond_cat)
    by["condition"] = by["condition"].astype(cond_cat)
    overall = overall.sort_values(["metric","stratum","condition"]).reset_index(drop=True)
    by = by.sort_values(["metric","complexity_level","condition"]).reset_index(drop=True)

    paired = paired.reset_index(drop=True)
    return overall, by, paired

tokens_overall, tokens_by_complexity, tokens_paired = efficiency_tables("tokens", "tokens")
ttft_overall, ttft_by_complexity, ttft_paired = efficiency_tables("TTFT", "TTFT_ms")


## Step 5 — Error categories (descriptive only) and transition patterns

Error categories are standardized to the final set:
- **structural / semantic / computational / none**

We report descriptive summaries only:
- counts and percentages by condition (overall; optionally by complexity)

We also export transition summaries across raw → struct → sem:
- correctness transitions (incorrect→incorrect→correct, etc.)
- error-category transitions (e.g., structural→semantic→none)

No inferential tests (chi-square/Fisher) are performed for error-category distributions.


In [26]:
def error_counts(df_in: pd.DataFrame):
    rows = []
    strata = [("overall", df_in)] + [(f"complexity_{int(l)}", g) for l,g in df_in.groupby("complexity_level", sort=True)]
    for stratum, g in strata:
        n = len(g)
        for c in CONDS:
            vc = g[f"error_category_{c}"].astype(str).str.strip().str.lower().value_counts()
            for cat, cnt in vc.items():
                rows.append({
                    "stratum": stratum, "condition": c, "error_category": cat,
                    "count": int(cnt), "percent": 100*float(cnt/n), "n_questions": n
                })
    out = pd.DataFrame(rows)
    out["condition"] = out["condition"].astype(cond_cat)
    out = out.sort_values(["stratum","condition","error_category"]).reset_index(drop=True)
    return out

errors_counts = error_counts(df)

# transitions per question using canonical status (correct vs error_category)
trans = pd.DataFrame({"question_id": df["question_id"], "complexity_level": df["complexity_level"].astype(int)})
for c in CONDS:
    trans[f"status_{c}"] = np.where(df[f"correct_{c}"].astype(int)==1, "none", df[f"error_category_{c}"].astype(str).str.strip().str.lower())

trans["transition_errorcat"] = trans["status_raw"] + " -> " + trans["status_struct"] + " -> " + trans["status_sem"]
trans["transition_correctness"] = (
    np.where(df["correct_raw"].astype(int)==1, "correct", "incorrect") + " -> " +
    np.where(df["correct_struct"].astype(int)==1, "correct", "incorrect") + " -> " +
    np.where(df["correct_sem"].astype(int)==1, "correct", "incorrect")
)

transitions_summary_errorcat = trans["transition_errorcat"].value_counts().reset_index()
transitions_summary_errorcat.columns = ["transition_errorcat", "count"]
transitions_summary_errorcat["percent"] = 100*transitions_summary_errorcat["count"]/len(trans)

transitions_summary_correctness = trans["transition_correctness"].value_counts().reset_index()
transitions_summary_correctness.columns = ["transition_correctness", "count"]
transitions_summary_correctness["percent"] = 100*transitions_summary_correctness["count"]/len(trans)

display(errors_counts.head())
display(trans.head())
display(transitions_summary_correctness)
display(transitions_summary_errorcat)


,stratum,condition,error_category,count,percent,n_questions
0,complexity_1,raw,none,5,100.0,5
1,complexity_1,struct,none,5,100.0,5
2,complexity_1,sem,none,5,100.0,5
3,complexity_2,raw,none,1,20.0,5
4,complexity_2,raw,semantic,3,60.0,5


,question_id,complexity_level,status_raw,status_struct,status_sem,transition_errorcat,transition_correctness
0,1,1,none,none,none,none -> none -> none,correct -> correct -> correct
1,2,2,semantic,semantic,none,semantic -> semantic -> none,incorrect -> incorrect -> correct
2,3,3,computational,structural,none,computational -> structural -> none,incorrect -> incorrect -> correct
3,4,4,none,none,none,none -> none -> none,correct -> correct -> correct
4,5,5,structural,structural,none,structural -> structural -> none,incorrect -> incorrect -> correct


,transition_correctness,count,percent
0,incorrect -> incorrect -> correct,14,56.0
1,correct -> correct -> correct,8,32.0
2,incorrect -> correct -> correct,2,8.0
3,incorrect -> incorrect -> incorrect,1,4.0


,transition_errorcat,count,percent
0,none -> none -> none,8,32.0
1,semantic -> semantic -> none,5,20.0
2,structural -> semantic -> none,5,20.0
3,structural -> structural -> none,3,12.0
4,computational -> structural -> none,1,4.0
5,structural -> structural -> structural,1,4.0
6,computational -> none -> none,1,4.0
7,structural -> none -> none,1,4.0


## Step 6 — Process measures and mechanism links

**Stage-1 process measures (per condition):**
For each binary process variable (yes/no; NA excluded), we report:
- proportion + Wilson 95% CI (with classifiable denominator)

**Between-condition comparisons (descriptive):**
Paired McNemar tests are run on questions that are classifiable in both conditions (paired complete-case).
We report:
- paired N
- discordants labeled as improved/worsened (A=no,B=yes / A=yes,B=no)
- p-value (no multiplicity correction)

**Stage-2 conditional measures (descriptive only):**
- P(source_correct=yes | structural_assumption=yes)
- P(interpretation_correct=yes | semantic_assumption=yes)
with numerator/denominator and **Clopper–Pearson** 95% CI.

**Mechanism links to end-to-end correctness (exploratory):**
Within each condition, we compute 2×2 tables:
- correctness vs source_correct
- correctness vs interpretation_correct
and run Fisher’s exact test (two-sided), reporting p-values and cell counts.


In [27]:
process_vars = [
    ("structural_assumption_made", "Structural assumption rate"),
    ("semantic_assumption_made", "Semantic assumption rate"),
    ("source_correct", "Correct source/column rate"),
    ("interpretation_correct", "Correct interpretation rate"),
]

# rates + normalized cache (strip+lower happens inside norm_yes_no_na)
rate_rows = []
norm_cache = {}

for base, label in process_vars:
    for c in CONDS:
        col = f"{base}_{c}"
        s_norm = norm_yes_no_na(df[col])
        norm_cache[(base,c)] = s_norm
        r = rate_ci(s_norm)
        rate_rows.append({"process_variable": base, "label": label, "condition": c, **r})

process_rates = pd.DataFrame(rate_rows)
process_rates["condition"] = process_rates["condition"].astype(cond_cat)
process_rates = process_rates.sort_values(["process_variable","condition"]).reset_index(drop=True)

# Stage-1 paired McNemar (descriptive; no multiplicity correction)
mcn_rows = []
for base, label in process_vars:
    for a,b in PAIRS:
        res = paired_mcnemar_process(norm_cache[(base,a)], norm_cache[(base,b)])
        mcn_rows.append({
            "process_variable": base,
            "label": label,
            "comparison": f"{a}_vs_{b}",
            "paired_n": res["n_pairs"],
            "improved": res["improved"],   # A=no, B=yes
            "worsened": res["worsened"],   # A=yes, B=no
            "p_value": res["mcnemar_p"],
            "used_exact": res["used_exact"],
        })
process_paired_mcnemar = pd.DataFrame(mcn_rows)

# Mechanism links to correctness (exploratory): Fisher exact within-condition
link_rows = []
for c in CONDS:
    for base in ["source_correct", "interpretation_correct"]:
        link = link_2x2_fisher(df[f"correct_{c}"], norm_cache[(base,c)])
        link_rows.append({"condition": c, "process_variable": base, **link})
process_links = pd.DataFrame(link_rows)
process_links["condition"] = process_links["condition"].astype(cond_cat)
process_links = process_links.sort_values(["process_variable","condition"]).reset_index(drop=True)

# Stage-2 conditional rates (descriptive only) with Clopper–Pearson CI
def conditional_rate_cp(numer_mask: pd.Series, denom_mask: pd.Series):
    denom = int(denom_mask.sum())
    if denom == 0:
        return {"denom_n":0,"num_x":0,"rate":np.nan,"cp95_low":np.nan,"cp95_high":np.nan}
    x = int((numer_mask & denom_mask).sum())
    lo, hi = clopper_pearson_ci(x, denom)
    return {"denom_n":denom,"num_x":x,"rate":x/denom,"cp95_low":lo,"cp95_high":hi}

stage2_rows = []
for c in CONDS:
    sa = norm_cache[("structural_assumption_made",c)]
    sc = norm_cache[("source_correct",c)]
    denom = (sa=="yes") & sc.isin(["yes","no"])
    numer = (sc=="yes")
    stage2_rows.append({
        "condition": c,
        "conditional": "P(source_correct=yes | structural_assumption=yes)",
        **conditional_rate_cp(numer, denom)
    })

    se = norm_cache[("semantic_assumption_made",c)]
    ic = norm_cache[("interpretation_correct",c)]
    denom = (se=="yes") & ic.isin(["yes","no"])
    numer = (ic=="yes")
    stage2_rows.append({
        "condition": c,
        "conditional": "P(interpretation_correct=yes | semantic_assumption=yes)",
        **conditional_rate_cp(numer, denom)
    })

stage2_conditionals = pd.DataFrame(stage2_rows)
stage2_conditionals["condition"] = stage2_conditionals["condition"].astype(cond_cat)
stage2_conditionals = stage2_conditionals.sort_values(["conditional","condition"]).reset_index(drop=True)

display(process_rates)
display(process_paired_mcnemar)
display(process_links)
display(stage2_conditionals)


,process_variable,label,condition,n_classifiable,n_yes,rate,wilson95_low,wilson95_high
0,interpretation_correct,Correct interpretation rate,raw,11,5,0.454545,0.212713,0.719908
1,interpretation_correct,Correct interpretation rate,struct,16,5,0.312500,0.141646,0.555956
2,interpretation_correct,Correct interpretation rate,sem,21,21,1.000000,0.845361,1.000000
3,semantic_assumption_made,Semantic assumption rate,raw,25,19,0.760000,0.565703,0.885037
4,semantic_assumption_made,Semantic assumption rate,struct,25,19,0.760000,0.565703,0.885037
5,semantic_assumption_made,Semantic assumption rate,sem,25,10,0.400000,0.234033,0.592605
6,source_correct,Correct source/column rate,raw,22,12,0.545455,0.346598,0.730797
7,source_correct,Correct source/column rate,struct,22,15,0.681818,0.473186,0.836394
8,source_correct,Correct source/column rate,sem,22,21,0.954545,0.782020,0.991931
9,structural_assumption_made,Structural assumption rate,raw,25,12,0.480000,0.300313,0.665015


,process_variable,label,comparison,paired_n,improved,worsened,p_value,used_exact
0,structural_assumption_made,Structural assumption rate,raw_vs_struct,25,1,7,0.070312,True
1,structural_assumption_made,Structural assumption rate,struct_vs_sem,25,0,5,0.062500,True
2,structural_assumption_made,Structural assumption rate,raw_vs_sem,25,0,11,0.000977,True
3,semantic_assumption_made,Semantic assumption rate,raw_vs_struct,25,1,1,1.000000,True
4,semantic_assumption_made,Semantic assumption rate,struct_vs_sem,25,1,10,0.011719,True
5,semantic_assumption_made,Semantic assumption rate,raw_vs_sem,25,2,11,0.022461,True
6,source_correct,Correct source/column rate,raw_vs_struct,22,4,1,0.375000,True
7,source_correct,Correct source/column rate,struct_vs_sem,22,6,0,0.031250,True
8,source_correct,Correct source/column rate,raw_vs_sem,22,9,0,0.003906,True
9,interpretation_correct,Correct interpretation rate,raw_vs_struct,10,0,0,1.000000,True


,condition,process_variable,n,table_proc_no_incorrect,table_proc_no_correct,table_proc_yes_incorrect,table_proc_yes_correct,method,p_correct_given_proc_yes,p_correct_given_proc_no,p_value
0,raw,interpretation_correct,11,6,0,0,5,fisher_exact_2x2,1.000000,0.0,0.002165
1,struct,interpretation_correct,16,11,0,0,5,fisher_exact_2x2,1.000000,0.0,0.000229
2,sem,interpretation_correct,21,0,0,0,21,skipped_degenerate_table,1.000000,NaN,NaN
3,raw,source_correct,22,10,0,7,5,fisher_exact_2x2,0.416667,0.0,0.039645
4,struct,source_correct,22,7,0,8,7,fisher_exact_2x2,0.466667,0.0,0.051283
5,sem,source_correct,22,1,0,0,21,fisher_exact_2x2,1.000000,0.0,0.045455


,condition,conditional,denom_n,num_x,rate,cp95_low,cp95_high
0,raw,P(interpretation_correct=yes | semantic_assump...,9,4,0.444444,0.136996,0.787991
1,struct,P(interpretation_correct=yes | semantic_assump...,14,3,0.214286,0.046579,0.507976
2,sem,P(interpretation_correct=yes | semantic_assump...,9,9,1.000000,0.663733,1.000000
3,raw,P(source_correct=yes | structural_assumption=yes),12,2,0.166667,0.020863,0.484138
4,struct,P(source_correct=yes | structural_assumption=yes),6,2,0.333333,0.043272,0.777222
5,sem,P(source_correct=yes | structural_assumption=yes),1,0,0.000000,0.000000,0.975000


## Step 7 — Export canonical tables and logs

We export canonical CSVs to `outputs/.../analysis/tables/` for consumption by Notebook 05.
We also write QA summaries and a run manifest to `outputs/.../analysis/logs/`.

In [28]:
# ------------------------------------------------------------
# Canonical per-question table (for distribution plots in 05)
# ------------------------------------------------------------
per_question = df[["question_id", "complexity_level"]].copy()

# correctness + tokens + TTFT (per question, per condition)
for c in CONDS:
    per_question[f"correct_{c}"] = df[f"correct_{c}"].astype(int)
    per_question[f"tokens_{c}"]  = df[f"tokens_{c}"].astype(float)
    per_question[f"TTFT_{c}"]    = df[f"TTFT_{c}"].astype(float)
    per_question[f"error_category_{c}"] = df[f"error_category_{c}"].astype(str)

# add status per condition (none if correct else error_category)
for c in CONDS:
    per_question[f"status_{c}"] = np.where(
        per_question[f"correct_{c}"]==1,
        "none",
        per_question[f"error_category_{c}"].astype(str).str.strip().str.lower()
    )

# add transition label (raw -> struct -> sem)
per_question["transition"] = (
    per_question["status_raw"] + " -> " +
    per_question["status_struct"] + " -> " +
    per_question["status_sem"]
)

display(per_question.head())


,question_id,complexity_level,correct_raw,tokens_raw,TTFT_raw,error_category_raw,correct_struct,tokens_struct,TTFT_struct,error_category_struct,correct_sem,tokens_sem,TTFT_sem,error_category_sem,status_raw,status_struct,status_sem,transition
0,1,1,1,80293.0,2205.0,none,1,12597.0,981.0,none,1,13601.0,1816.0,none,none,none,none,none -> none -> none
1,2,2,0,243787.0,1831.0,semantic,0,21017.0,946.0,semantic,1,21292.0,630.0,none,semantic,semantic,none,semantic -> semantic -> none
2,3,3,0,80829.0,2142.0,computational,0,33629.0,1055.0,structural,1,36170.0,1086.0,none,computational,structural,none,computational -> structural -> none
3,4,4,1,252264.0,2120.0,none,1,42406.0,1075.0,none,1,68603.0,3456.0,none,none,none,none,none -> none -> none
4,5,5,0,305055.0,2684.0,structural,0,48607.0,3138.0,structural,1,101227.0,994.0,none,structural,structural,none,structural -> structural -> none


In [29]:
# canonical tables (consumed by Notebook 05)accuracy_overall.to_csv(OUT_TAB / "accuracy_overall.csv", index=False)
accuracy_by_complexity.to_csv(OUT_TAB / "accuracy_by_complexity.csv", index=False)
mcnemar_results.to_csv(OUT_TAB / "accuracy_mcnemar_holm.csv", index=False)
bootstrap_risk_diffs.to_csv(OUT_TAB / "accuracy_bootstrap_risk_diffs.csv", index=False)

tokens_overall.to_csv(OUT_TAB / "tokens_overall.csv", index=False)
tokens_by_complexity.to_csv(OUT_TAB / "tokens_by_complexity.csv", index=False)
tokens_paired.to_csv(OUT_TAB / "tokens_paired_bootstrap.csv", index=False)

ttft_overall.to_csv(OUT_TAB / "ttft_overall.csv", index=False)
ttft_by_complexity.to_csv(OUT_TAB / "ttft_by_complexity.csv", index=False)
ttft_paired.to_csv(OUT_TAB / "ttft_paired_bootstrap.csv", index=False)

errors_counts.to_csv(OUT_TAB / "errors_counts.csv", index=False)
trans.to_csv(OUT_TAB / "transitions_per_question.csv", index=False)
transitions_summary_correctness.to_csv(OUT_TAB / "transitions_summary_correctness.csv", index=False)
transitions_summary_errorcat.to_csv(OUT_TAB / "transitions_summary_errorcat.csv", index=False)

process_rates.to_csv(OUT_TAB / "process_rates.csv", index=False)
process_paired_mcnemar.to_csv(OUT_TAB / "process_paired_mcnemar.csv", index=False)
process_links.to_csv(OUT_TAB / "process_links_2x2_fisher.csv", index=False)
stage2_conditionals.to_csv(OUT_TAB / "stage2_conditionals_cp.csv", index=False)


# QA token totals
qa_token_totals = pd.DataFrame({
    "condition": COND_ORDER,
    "token_total_sum": [float(df[f"tokens_{c}"].sum()) for c in COND_ORDER],
    "n_questions": [int(df[f"tokens_{c}"].notna().sum()) for c in COND_ORDER],
})
qa_token_totals.to_csv(OUT_LOG / "qa_token_totals.csv", index=False)

manifest = {
    "paper_slug": PAPER_SLUG,
    "release_id": RELEASE_ID,
    "run_id": RUN_ID,
    "created_at": NOW,
    "input_path": WIDE_PATH,
    "input_sha256": sha256_file(WIDE_PATH),
    "settings": {"seed": SEED, "n_boot": N_BOOT, "unit_of_analysis": "question (N=25)"},
    "outputs_dir": str(BASE_OUT),
    "tables_written": [p.name for p in sorted(OUT_TAB.glob("*.csv"))],
    "logs_written": [p.name for p in sorted(OUT_LOG.glob("*.csv"))],
}

with open(OUT_LOG / f"{PAPER_SLUG}_{RELEASE_ID}_{RUN_ID}_manifest_04.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Canonical analysis tables written to:", OUT_TAB)
print("Logs written to:", OUT_LOG)


Canonical analysis tables written to: ../outputs/buddi_paper/v1/analysis/tables
Logs written to: ../outputs/buddi_paper/v1/analysis/logs


In [30]:
# Additional check for values per-question-metrics
# print("IN-MEMORY per_question status unique values:")
# for c in ["raw","struct","sem"]:
#     col = f"status_{c}"
#     print(col, sorted(per_question[col].astype(str).str.strip().str.lower().unique()))

# out_path = OUT_TAB / "per_question_metrics.csv"
# print("Will write to:", out_path.resolve())
# print("About to write...")

# per_question.to_csv(out_path, index=False)

# print("Wrote. File mtime:", pd.to_datetime(out_path.stat().st_mtime, unit="s"))
# print("Reloading from disk to confirm...")
# tmp = pd.read_csv(out_path)

# for c in ["raw","struct","sem"]:
#     col = f"status_{c}"
#     print("ON-DISK", col, sorted(tmp[col].astype(str).str.strip().str.lower().unique()))
